In [5]:
import pandas as pd
import numpy as np

df_train=pd.read_csv("train (1).csv")
df_test=pd.read_csv("test (1).csv")

df_train.head()

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
0,0,785.0,NaN,24472.0,F,NaN,NaN,NaN,N,0.8,NaN,3.01,NaN,NaN,NaN,NaN,80.0,11.1,4.0,D
1,1,1639.0,NaN,18628.0,F,NaN,NaN,NaN,N,0.8,NaN,3.33,NaN,NaN,NaN,NaN,273.0,10.4,3.0,C
2,2,1095.0,NaN,21185.0,F,NaN,NaN,NaN,N,0.6,NaN,3.76,NaN,NaN,NaN,NaN,312.0,10.0,4.0,C
3,3,1581.0,NaN,24472.0,F,NaN,NaN,NaN,N,1.0,NaN,3.48,NaN,NaN,NaN,NaN,277.0,10.0,2.0,C
4,4,3222.0,Placebo,18713.0,F,N,Y,Y,Y,2.5,408.0,3.70,145.0,856.0,110.05,98.0,132.0,11.0,4.0,D


In [6]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             15000 non-null  int64  
 1   N_Days         15000 non-null  float64
 2   Drug           8365 non-null   str    
 3   Age            15000 non-null  float64
 4   Sex            15000 non-null  str    
 5   Ascites        8370 non-null   str    
 6   Hepatomegaly   8361 non-null   str    
 7   Spiders        8355 non-null   str    
 8   Edema          15000 non-null  str    
 9   Bilirubin      15000 non-null  float64
 10  Cholesterol    6510 non-null   float64
 11  Albumin        15000 non-null  float64
 12  Copper         8253 non-null   float64
 13  Alk_Phos       8350 non-null   float64
 14  SGOT           8350 non-null   float64
 15  Tryglicerides  6463 non-null   float64
 16  Platelets      14408 non-null  float64
 17  Prothrombin    14980 non-null  float64
 18  Stage          15

In [7]:
df_train.isna().sum()

id                  0
N_Days              0
Drug             6635
Age                 0
Sex                 0
Ascites          6630
Hepatomegaly     6639
Spiders          6645
Edema               0
Bilirubin           0
Cholesterol      8490
Albumin             0
Copper           6747
Alk_Phos         6650
SGOT             6650
Tryglicerides    8537
Platelets         592
Prothrombin        20
Stage               0
Status              0
dtype: int64

In [8]:
cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Stage']

for col in cat_cols:
    df_train[col] = df_train[col].fillna('Missing').astype(str)
    df_test[col] = df_test[col].fillna('Missing').astype(str)

In [9]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

print("📊 Важность признаков:")
print(feature_importance)

AttributeError: 'XGBClassifier' object has no attribute 'get_feature_importance'

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# 1. Загрузка данных
df_train = pd.read_csv("train (1).csv")
df_test = pd.read_csv("test (1).csv")

# 2. Маппинг целевой переменной
target_map = {'C': 0, 'CL': 1, 'D': 2}
y = df_train['Status'].map(target_map)

# 3. Разделение на признаки
X = df_train.drop(columns=['id', 'Status']).copy()
X_test = df_test.drop(columns=['id']).copy()

# ==========================================
# 4. БАЗОВЫЙ FEATURE ENGINEERING (Безопасный)
# ==========================================
for data in [X, X_test]:
    # Считаем пропуски
    data['null_count'] = data.isna().sum(axis=1)
    
    # Соотношения показателей (добавляем 1e-5 от деления на ноль)
    data['Bilirubin_Albumin_ratio'] = data['Bilirubin'] / (data['Albumin'] + 1e-5)
    data['Copper_Bilirubin'] = data['Copper'] * data['Bilirubin']

# Категориальные колонки
cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Stage']
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)

# ==========================================
# 5. K-FOLD С ПРАВИЛЬНЫМ РАСЧЕТОМ СТАТИСТИК
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros((len(X_test), 3))
cv_scores = []

num_target_cols = ['Bilirubin', 'Copper', 'Prothrombin', 'Platelets']

print("🚀 Запуск 5-Fold CV...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    # Выборка для текущего фолда
    X_train_fold = X.iloc[train_idx].copy()
    y_train_fold = y.iloc[train_idx]
    
    X_val_fold = X.iloc[val_idx].copy()
    y_val_fold = y.iloc[val_idx]  # <-- Исправлено: теперь y_val_fold объявлен!
    
    X_test_fold = X_test.copy()
    
    # Групповые агрегации считаем строго на X_train_fold (без утечек)
    for col in num_target_cols:
        stage_means = X_train_fold.groupby('Stage')[col].mean()
        
        for df_curr in [X_train_fold, X_val_fold, X_test_fold]:
            df_curr[f'{col}_stage_mean'] = df_curr['Stage'].map(stage_means)
            df_curr[f'{col}_diff_stage'] = df_curr[col] - df_curr[f'{col}_stage_mean']
    
    # Создаем модель с адекватными параметрами
    model = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.03,
        depth=6,                      # Возвращаем надежную глубину 6
        l2_leaf_reg=3,
        cat_features=cat_cols,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        random_seed=42 + fold,
        verbose=0
    )
    
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=(X_val_fold, y_val_fold),
        early_stopping_rounds=100,
        verbose=False
    )
    
    val_preds = model.predict_proba(X_val_fold)
    score = log_loss(y_val_fold, val_preds)
    cv_scores.append(score)
    
    print(f"Fold {fold + 1} | Best Iter: {model.get_best_iteration()} | Log Loss: {score:.5f}")
    
    # Суммируем предсказания для теста
    test_preds += model.predict_proba(X_test_fold) / 5

mean_cv = np.mean(cv_scores)
print("\n" + "="*35)
print(f"🏆 Честный Средний CV Log Loss: {mean_cv:.5f}")
print("="*35)

# Сохранение сабмита
submission = pd.DataFrame({
    'id': df_test['id'],
    'Status_C': test_preds[:, 0],
    'Status_CL': test_preds[:, 1],
    'Status_D': test_preds[:, 2]
})
submission.to_csv('submission_tuned.csv', index=False)
print("Файл 'submission_tuned.csv' успешно обновлен!")

🚀 Запуск 5-Fold CV...
Fold 1 | Best Iter: 1207 | Log Loss: 0.37221
Fold 2 | Best Iter: 1449 | Log Loss: 0.38550
Fold 3 | Best Iter: 1400 | Log Loss: 0.38439
Fold 4 | Best Iter: 1450 | Log Loss: 0.37789
Fold 5 | Best Iter: 1443 | Log Loss: 0.36725

🏆 Честный Средний CV Log Loss: 0.37745
Файл 'submission_tuned.csv' успешно обновлен!


: 

In [10]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# 1. Загрузка данных
df_train = pd.read_csv("train (1).csv")
df_test = pd.read_csv("test (1).csv")

# 2. Маппинг целевой переменной
target_map = {'C': 0, 'CL': 1, 'D': 2}
y = df_train['Status'].map(target_map)

X = df_train.drop(columns=['id', 'Status']).copy()
X_test = df_test.drop(columns=['id']).copy()

# 3. Базовый Feature Engineering
for data in [X, X_test]:
    data['null_count'] = data.isna().sum(axis=1)
    data['Bilirubin_Albumin_ratio'] = data['Bilirubin'] / (data['Albumin'] + 1e-5)
    data['Copper_Bilirubin'] = data['Copper'] * data['Bilirubin']

# 4. Кодируем категории в ЧИСЛА
cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Stage']
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)
    
    X[col] = X[col].astype('category').cat.codes
    X_test[col] = X_test[col].astype('category').cat.codes

# 5. K-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_preds = np.zeros((len(X_test), 3))
cv_scores = []

num_target_cols = ['Bilirubin', 'Copper', 'Prothrombin', 'Platelets']

print("🚀 Запуск XGBoost 5-Fold CV...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold = X.iloc[train_idx].copy()
    y_train_fold = y.iloc[train_idx]
    
    X_val_fold = X.iloc[val_idx].copy()
    y_val_fold = y.iloc[val_idx]
    
    X_test_fold = X_test.copy()
    
    # Групповые агрегации
    for col in num_target_cols:
        stage_means = X_train_fold.groupby('Stage', observed=False)[col].mean()
        
        for df_curr in [X_train_fold, X_val_fold, X_test_fold]:
            df_curr[f'{col}_stage_mean'] = df_curr['Stage'].map(stage_means).astype(float)
            df_curr[f'{col}_diff_stage'] = df_curr[col] - df_curr[f'{col}_stage_mean']
    
    # 6. Инициализация XGBoost с early_stopping_rounds
    model = XGBClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        early_stopping_rounds=100,  # <-- ДОБАВИЛИ ОБРАТНО
        random_state=42 + fold
    )
    
    # Обучение
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=False
    )
    
    val_preds = model.predict_proba(X_val_fold)
    score = log_loss(y_val_fold, val_preds)
    cv_scores.append(score)
    
    # Теперь model.best_iteration доступен и не вызовет ошибку
    print(f"Fold {fold + 1} | Best Iter: {model.best_iteration} | Log Loss: {score:.5f}")
    
    xgb_preds += model.predict_proba(X_test_fold) / 5

mean_cv = np.mean(cv_scores)
print("\n" + "="*35)
print(f"🏆 Средний CV Log Loss (XGBoost): {mean_cv:.5f}")
print("="*35)

# Сохранение предсказаний
submission = pd.DataFrame({
    'id': df_test['id'],
    'Status_C': xgb_preds[:, 0],
    'Status_CL': xgb_preds[:, 1],
    'Status_D': xgb_preds[:, 2]
})
submission.to_csv('submission_xgb.csv', index=False)
print("Файл 'submission_xgb.csv' успешно создан!")

🚀 Запуск XGBoost 5-Fold CV...
Fold 1 | Best Iter: 470 | Log Loss: 0.36688
Fold 2 | Best Iter: 458 | Log Loss: 0.37997
Fold 3 | Best Iter: 438 | Log Loss: 0.37499
Fold 4 | Best Iter: 447 | Log Loss: 0.37736
Fold 5 | Best Iter: 478 | Log Loss: 0.36326

🏆 Средний CV Log Loss (XGBoost): 0.37249
Файл 'submission_xgb.csv' успешно создан!
